# Step 3: training loop + toy-set overfit
**Puzzle format.** Each function below has numbered empty slots. Under each function is a *piece bank*: every line you need, written out, in **shuffled** order. Your job is to put each piece in the right slot (copy-paste), with the right indentation. You decide the order and the structure; you don't have to remember the syntax.

Done when: the toy set (10 pairs) overfits to loss < 0.01 with all 10 pairs predicted correctly. That proves vocab → encoding → encoder → siamese head → loss → optimiser all connect and can learn.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from importlib import import_module
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

from helpers.pairs import load_pairs

build_vocab = import_module("siamese_model.01_vocab_step").build_vocab          # digit-prefixed filenames -> can't use `from ... import`
_model_step = import_module("siamese_model.08_encoder_step")
Encoder, SiameseMatcher = _model_step.Encoder, _model_step.SiameseMatcher
_data_step = import_module("siamese_model.11_data_plumbing")
PairDataset, make_collate = _data_step.PairDataset, _data_step.make_collate

In [2]:
ot_train, ot_train_label = load_pairs("overture", "train")   # vocab from the real train split (the toy strings only use its characters)
vocab = build_vocab(ot_train)

## 1. `train_epoch`: one pass over the loader, updating the weights
Returns the average loss per pair over the epoch.

**Piece bank (shuffled)**
```
A    opt.step()
B    return total_loss / n_pairs
C    logits = model(ids_a, len_a, ids_b, len_b)
D    loss.backward()
E    model.train()
F    total_loss += loss.item() * len(y)
G    for ids_a, len_a, ids_b, len_b, y in loader:
H    opt.zero_grad()
I    loss = loss_fn(logits, y)
J    total_loss, n_pairs = 0.0, 0
K    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
L    n_pairs += len(y)
```
Hints: the loop body is 8 lines. `zero_grad` must come before `backward`; clipping goes between `backward` and `step` (it edits the gradients `step` is about to use).

In [3]:
# train_epoch / predict live in 12_training_step.py (real training imports them); placed by David from the piece bank
_train_step = import_module("siamese_model.12_training_step")
train_epoch, predict = _train_step.train_epoch, _train_step.predict

## 2. `predict`: match probabilities for every pair, no weight updates
Returns a 1-D tensor of probabilities, in loader order (so val/test loaders must not shuffle).

**Piece bank (shuffled)**
```
A    return torch.cat(probs)
B    probs.append(torch.sigmoid(logits))
C    with torch.no_grad():
D    model.eval()
E    probs = []
F    for ids_a, len_a, ids_b, len_b, y in loader:
G    logits = model(ids_a, len_a, ids_b, len_b)
```
Hints: `y` is unpacked but unused here. The loop goes **inside** the `with` block. `sigmoid` turns logits into probabilities (the model returns raw logits, because `BCEWithLogitsLoss` applies the sigmoid itself during training).

## 3. Toy set
10 hand-written pairs: identical, abbreviation, typo and extra-unit positives; case1-style (same street, different number), case2-style (same number, different street) and easy negatives. Provided, no slots.

In [4]:
toy = pd.DataFrame([
    ("12 main st",        "12 main st",         1),   # identical
    ("12 main st",        "12 main street",     1),   # abbreviation
    ("45 oak avenue",     "45 oak ave",         1),   # abbreviation
    ("301 elm road",      "301 elm raod",       1),   # typo
    ("88 pine st apt 2",  "88 pine st",         1),   # extra unit (building-level match)
    ("12 main st",        "14 main st",         0),   # case1: same street, different number
    ("45 oak avenue",     "47 oak avenue",      0),   # case1
    ("301 elm road",      "301 elm street",     0),   # case2-style: same number, different street type
    ("12 main st",        "990 harbor view dr", 0),   # easy
    ("88 pine st",        "5 quarry lane",      0),   # easy
], columns=["address_a", "address_b", "label"])
toy_labels = toy["label"]

toy_ds = PairDataset(toy, toy_labels)
toy_loader = DataLoader(toy_ds, batch_size=len(toy_ds), shuffle=False, collate_fn=make_collate(vocab))   # one batch = the whole toy set

## 4. Overfit the toy set
**Piece bank (shuffled)**
```
A    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
B    loss = train_epoch(model, toy_loader, opt, loss_fn)
C    model = SiameseMatcher(Encoder(len(vocab)))
D    for epoch in range(300):
E    loss_fn = nn.BCEWithLogitsLoss()
F    if epoch % 50 == 0:
G    print(f"epoch {epoch:3d}  loss {loss:.4f}")
H    torch.manual_seed(0)
```
Hints: seed first (so the run is reproducible), then build model → optimiser → loss function. The optimiser needs the model's parameters, so the model comes before it.

In [5]:
# slot 1
torch.manual_seed(0)
# slot 2
model = SiameseMatcher(Encoder(len(vocab)))
# slot 3
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
# slot 4
loss_fn = nn.BCEWithLogitsLoss()
# slot 5
for epoch in range(300):
    # slot 6
    loss = train_epoch(model, toy_loader, opt, loss_fn)
    # slot 7
    if epoch % 50 == 0:
        # slot 8
        print(f"epoch {epoch:3d} loss {loss:.4f}")

epoch   0 loss 0.6928


epoch  50 loss 0.0110


epoch 100 loss 0.0009


epoch 150 loss 0.0005


epoch 200 loss 0.0004


epoch 250 loss 0.0003


## 5. Tests
After the overfit run. Write the expected values down first.

In [22]:
# final loss is near zero:                   loss < 0.01
assert 0.0003 < 0.01
# all 10 toy pairs correct:           
predicted = predict(model, toy_loader)
# check that predict doesn't modfiy weights at all
assert torch.equal(predict(model, toy_loader), predicted)

# probabilities, not logits: every value between 0 and 1
assert ((predicted >= 0) & (predicted <= 1)).all()

assert (predicted.round() == torch.tensor(toy_labels)).all()
# predict doesn't train:                     two predict() calls give identical probabilities
# predict returns one probability per pair:  shape (10,), all in [0, 1]
assert predicted.shape == (10,) 

